# `semidde` API quickstart: `fit()` and `select_hyperparameters()`

This notebook is a minimal interface demo. It shows two common entry points:

- `fit()`: fit one specified DDE-family model;
- `select_hyperparameters()`: evaluate candidate hyperparameter settings and optionally refit the best model.

The examples use tiny simulated ordinal data so the notebook runs quickly. For real analyses, replace `Y` with your data matrix, keeping observations in rows and variables in columns.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Make the local source tree importable when running this notebook from scripts/.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "scripts":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "SemiDDE":
    REPO_ROOT = REPO_ROOT.parent

SRC = REPO_ROOT / "SemiDDE" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from semidde import fit, select_hyperparameters, copula_tau_default
from semidde.algorithms.copula import normal_initialize_copula_dde
from semidde.algorithms.copula import prepare_rank_likelihood_inputs
from semidde.simulation import generate_X_Cop_marg_emp_D

pd.set_option("display.max_colwidth", 90)

## 1. Create a small example dataset

`semidde` expects `Y` to be an `N x J` array: `N` observations by `J` variables. This example generates ordinal data from a small two-layer CopDDE-like model.

In [2]:
rng = np.random.default_rng(20260714)

N = 30       # number of observations
J = 18       # number of observed variables/items
D = 2        # number of latent layers
K_list = [6, 2]  # layer widths: 6 shallow latent nodes, 2 deep latent nodes

# Parameters used only to simulate demo data.
B1 = rng.normal(0, 0.35, size=(J, K_list[0] + 1))
B2 = rng.normal(0, 0.35, size=(K_list[0], K_list[1] + 1))
B1[:, 0] = 0.0  # intercept column
B2[:, 0] = 0.0
prop = np.full(K_list[-1], 0.45)
gamma = np.ones(J)

sim = generate_X_Cop_marg_emp_D(
    N,
    prop,
    lambdas=np.ones(J) * 2.0,
    gamma=gamma,
    B_list=[B1, B2],
    random_state=1,
)
Y = sim["X"]

print(Y.shape)
pd.DataFrame(Y).head()

(30, 18)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,0.0,5.0,1.0,0.0,3.0,2.0,2.0,1.0,4.0,0.0,2.0,2.0,1.0,2.0,1.0,2.0,4.0,1.0
1,4.0,2.0,3.0,1.0,5.0,1.0,0.0,5.0,0.0,0.0,4.0,0.0,2.0,2.0,3.0,1.0,2.0,2.0
2,0.0,9.0,1.0,1.0,8.0,2.0,0.0,3.0,3.0,0.0,3.0,2.0,1.0,5.0,2.0,1.0,1.0,1.0
3,0.0,6.0,3.0,2.0,6.0,2.0,0.0,4.0,4.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0,7.0,3.0
4,1.0,3.0,2.0,5.0,2.0,0.0,1.0,2.0,1.0,0.0,2.0,4.0,0.0,1.0,1.0,0.0,3.0,1.0


## 2. `fit()`: fit one chosen model

`fit()` is the direct model-fitting entry point. Use it when you already know the model class, dimensions, and tuning parameters.

Important arguments:

- `Y`: data matrix, rows are observations and columns are variables;
- `model`: `'semi'` for continuous SemiDDE, `'copula'` for CopDDE/rank-likelihood data, or `'auto'`;
- `K_list`: latent widths from shallow to deep, e.g. `[6, 2]` for two latent layers;
- `dimension_method`: `'fixed'` uses the supplied widths; `'csp'` enables CSP-based width selection for CopDDE;
- algorithm keywords such as `it`, `temp`, `tau`, `pen_list`, `t_spike`, and initialization inputs are passed through `**kwargs`.

In [3]:
# CopDDE uses rank-likelihood inputs. These are prepared from the observed ordinal data.
Z, R, Rlevels = prepare_rank_likelihood_inputs(Y)

# Data-based initialization for the latent states and loading matrices.
init = normal_initialize_copula_dde(Y, K_list, Z=Z, random_state=1)

fit_result = fit(
    Y,
    model="copula",             # use CopDDE for ordinal/mixed/rank-likelihood analysis
    dimension_method="fixed",   # use exactly K_list=[6, 2]
    K_list=K_list,

    # Rank-likelihood and initialization inputs.
    Z=Z,
    R=R,
    Rlevels=Rlevels,
    prop_in=init["prop"],
    gamma_in=init["gamma"],
    B_list_in=init["B_list"],
    A_list_in=init["A_list"],

    # Fast demo settings. Increase these for real analyses.
    C=1,                         # Monte Carlo sample count used inside SAEM updates
    tol=0.0,                     # convergence tolerance for fixed-dimension SAEM
    it=1,                        # SAEM iterations; e.g. 50 in the B5 analysis
    temp=0.8,                    # initial tempering temperature
    tau=0.1,                     # threshold for small loading values
    pen_list=[0.0, 0.0],         # fixed-dimension penalties, one per layer
    optimizer_maxiter=1,         # inner optimizer iteration budget
    verbose=False,
)

print(fit_result.model)
print([B.shape for B in fit_result.B_list])

CopDDE initialization: not triggered; using supplied initialized arguments.
copula
[(18, 7), (6, 3)]


## 3. `select_hyperparameters()`: choose tuning parameters and refit

`select_hyperparameters()` is the recommended entry point when you want the package to compare candidate settings.

Important arguments:

- `D`: number of latent layers;
- `model`: `'semi'` or `'copula'`;
- `dimension_method`: `'fixed'` or `'csp'` for CopDDE; SemiDDE currently supports `'fixed'`;
- `criterion`: `'bic'`, `'ebic'`, `'loglik'`, or `'pmse'`; `pmse` is available for CopDDE;
- `candidates`: list of dictionaries; each dictionary is one hyperparameter setting;
- `fixed_hp`: hyperparameters shared by all candidates;
- `base_fit_kwargs`: additional defaults used for every fit;
- `refit=True`: after selection, fit the best candidate on the full dataset and return that fitted result;
- `n_repeats` and `pmse_n_estimators`: pMSE-specific evaluation settings.

In [4]:
# Each dictionary is one candidate setting. Candidate values override fixed_hp/base defaults.
candidates = [
    {
        "name": "candidate_tau_0.05",
        "K_list": K_list,
        "C": 1,
        "tol": 0.0,
        "it": 1,
        "temp": 0.8,
        "pen_list": [0.0, 0.0],
        "tau": 0.05,
        "optimizer_maxiter": 1,
        "verbose": False,
    },
    {
        "name": "candidate_tau_0.10",
        "K_list": K_list,
        "C": 1,
        "tol": 0.0,
        "it": 1,
        "temp": 0.8,
        "pen_list": [0.0, 0.0],
        "tau": 0.10,
        "optimizer_maxiter": 1,
        "verbose": False,
    },
]

selected = select_hyperparameters(
    Y,
    D=D,
    model="copula",
    dimension_method="fixed",
    criterion="pmse",           # predictive pMSE: lower is better
    candidates=candidates,

    # Shared inputs used by every candidate.
    fixed_hp={
        "Z": Z,
        "R": R,
        "Rlevels": Rlevels,
        "prop_in": init["prop"],
        "gamma_in": init["gamma"],
        "B_list_in": init["B_list"],
        "A_list_in": init["A_list"],
    },

    # pMSE demo settings. For a real analysis, use more repeats and more trees.
    n_repeats=1,                 # B5 analysis used 50
    pmse_n_estimators=5,         # B5 analysis used 100
    refit=True,                  # return the best model refit on all rows
    random_state=1,
)

cv_table = pd.DataFrame(selected.diagnostics["cv_table"])
cv_table[["candidate_index", "name", "criterion", "mean_pmse", "std_pmse", "tau"]]

CopDDE initialization: not triggered; using supplied initialized arguments.
CopDDE initialization: not triggered; using supplied initialized arguments.


,candidate_index,name,criterion,mean_pmse,std_pmse,tau
0,0,candidate_tau_0.05,pmse,0.063333,0.0,0.05
1,1,candidate_tau_0.10,pmse,0.051333,0.0,0.10


## 4. CopDDE + CSP hyperparameter selection

For ordinal data with unknown latent width, use `dimension_method='csp'`. Then `K_list` is interpreted as the maximum width of each latent layer, and CSP shrinkage selects the active columns.

CSP-specific candidate arguments include:

- `t_spike`: spike scale(s), one per layer; smaller values encourage stronger shrinkage;
- `tau`: threshold for small loading entries;
- `temp`: initial tempering temperature;
- `it`, `optimizer_maxiter`, `convergence_std`: iteration and convergence controls.

In [7]:
csp_candidates = [
    {
        "name": "csp_light_shrinkage",
        "K_list": K_list,          # maximum widths, not necessarily final active widths
        "C": 1,
        "it": 1,
        "temp": 0.8,
        "tau": copula_tau_default(N),
        "t_spike": [0.02, 0.04],
        "optimizer_maxiter": 1,
        "convergence_std": 0.01,
        "verbose": False,
    },
    {
        "name": "csp_stronger_shrinkage",
        "K_list": K_list,
        "C": 1,
        "it": 1,
        "temp": 0.8,
        "tau": copula_tau_default(N),
        "t_spike": [0.04, 0.08],
        "optimizer_maxiter": 1,
        "convergence_std": 0.01,
        "verbose": False,
    },
]

selected_csp = select_hyperparameters(
    Y,
    D=D,
    model="copula",
    dimension_method="csp",
    criterion="pmse",
    candidates=csp_candidates,
    fixed_hp={
        "Z": Z,
        "R": R,
        "Rlevels": Rlevels,
        "prop_in": init["prop"],
        "gamma_in": init["gamma"],
        "B_list_in": init["B_list"],
        "A_list_in": init["A_list"],
    },
    n_repeats=1,
    pmse_n_estimators=5,
    refit=True,
    random_state=1,
)

pd.DataFrame(selected_csp.diagnostics["cv_table"])[[
    "candidate_index", "name", "criterion", "mean_pmse", "tau", "t_spike", "temp"
]]

CopDDE initialization: not triggered; using supplied initialized arguments.
CopDDE initialization: not triggered; using supplied initialized arguments.


,candidate_index,name,criterion,mean_pmse,tau,t_spike,temp
0,0,csp_light_shrinkage,pmse,0.064667,1.081396,"[0.02, 0.04]",0.8
1,1,csp_stronger_shrinkage,pmse,0.063333,1.081396,"[0.04, 0.08]",0.8


## 5. Inspect the returned result

Both `fit()` and `select_hyperparameters(..., refit=True)` return a `DDEResult`. Useful fields include:

- `result.B_list`: loading matrices, one per layer;
- `result.prop`: top-layer Bernoulli probabilities;
- `result.gamma`: copula residual variance/scaling parameters;
- `result.A_list`: estimated latent states when available;
- `result.diagnostics`: selection table, timing table, iteration metadata, and other run details.

In [8]:
result = selected_csp

print("Model:", result.model)
print("B shapes:", [B.shape for B in result.B_list])
print("prop shape:", result.prop.shape if result.prop is not None else None)
print("gamma shape:", result.gamma.shape if result.gamma is not None else None)
print("diagnostics keys:", sorted(result.diagnostics.keys()))

# Count nonzero active latent columns, excluding intercept columns.
active_by_layer = [int(np.any(np.asarray(B)[:, 1:] != 0, axis=0).sum()) for B in result.B_list]
print("Active latent columns by layer:", active_by_layer)

Model: copula
B shapes: [(18, 7), (6, 3)]
prop shape: (2,)
gamma shape: (18,)
diagnostics keys: ['A_sample_long', 'K_list', 'backend', 'cv_table', 'fold_scores', 'hyperparameter_selection', 'iterations', 'num_active', 'omega', 'p', 'theta', 'theta_slab', 'threshold_rule', 'timing_table', 'z_cusp']
Active latent columns by layer: [6, 2]


## 6. Practical template

For a real CopDDE+CSP analysis, the core call usually looks like this:

```python
result = select_hyperparameters(
    Y,
    D=2,
    model="copula",
    dimension_method="csp",
    criterion="pmse",
    candidates=my_grid,
    fixed_hp=my_shared_initialization,
    n_repeats=50,
    pmse_n_estimators=100,
    refit=True,
    random_state=20260714,
)
```

Use `fit()` when you want to fit one known setting; use `select_hyperparameters()` when you want to choose among settings.